In [ ]:
import datetime as dt

import contextily as cx
import geopandas as gpd
import pandas as pd
from meteora import clients, utils

In [ ]:
spatial_extent_filepath = "../data/processed/zurich-extent.gpkg"

# study period (e.g., JJA)
# # reference data to get study period
# ref_ts_df_filepath = "../data/raw/parallel-2025-int.csv"
# year = 2025
start_year = 2025
end_year = 2025
start_month = 6
end_month = 8

# output files
dst_ts_df_filepath = "../data/interim/zurich-lcd-ts-df.csv"
dst_stations_gdf_filepath = "../data/interim/zurich-lcd-stations.gpkg"

In [ ]:
region_gser = gpd.read_file(spatial_extent_filepath)["geometry"]

In [ ]:
client = clients.AWELClient(region_gser)
stations_gdf = client.stations_gdf.assign(source="AWEL").to_crs(region_gser.crs)
stations_gdf.index = stations_gdf.index.astype(str)

In [ ]:
ax = region_gser.plot(color="orange", edgecolor="k", alpha=0.5)
stations_gdf.plot("source", ax=ax, legend=True)
cx.add_basemap(ax, crs=region_gser.crs, source=cx.providers.CartoDB.Voyager)

In [ ]:
# # read reference data to get study period
# ref_ts_df = pd.read_csv(ref_ts_df_filepath, parse_dates=["time"]).set_index("time")

# # get data for study period
# ts_df = (
#     utils.long_to_wide(
#         client.get_ts_df("temperature", ref_ts_df.index[0], ref_ts_df.index[-1])
#     )
#     .resample("h")
#     .mean()
# )
# note that we need to transform each year data to wide and resample it before concat
# otherwise resample would add nan rows for all the missing non-JJA data
ts_df = pd.concat(
    [
        utils.long_to_wide(
            client.get_ts_df(
                "temperature",
                # pd.Timestamp(start),
                # dt.datetime.combine(
                #     pd.Timestamp(end),
                #     dt.time.max,
                # ),
                dt.date(year, start_month, 1),
                # get latest moment of the latest day of the month
                # ACHTUNG: this won't work if `end_month` is 12 (see commented code
                # below)
                dt.datetime.combine(
                    dt.date(year, end_month + 1, 1) - dt.timedelta(days=1),
                    dt.time.max,
                ),
            )
        )
        .resample("h")
        .mean()
        for year in range(start_year, end_year + 1)
    ]
)
ts_df.columns = ts_df.columns.astype(str)
ts_df.head()

In [ ]:
# save the time series of measurements to a file
ts_df.to_csv(dst_ts_df_filepath)
# save the stations' locations to a file
stations_gdf.to_file(dst_stations_gdf_filepath)